In [33]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import re


In [34]:
file_path = r"C:\Users\user\PycharmProjects\JupyterProject1\books\GoodReads_100k_books.csv"
data = pd.read_csv(file_path)

print("Original dataset shape:", data.shape)
print("\nColumns:")
print(data.columns.tolist())

Original dataset shape: (100000, 13)

Columns:
['author', 'bookformat', 'desc', 'genre', 'img', 'isbn', 'isbn13', 'link', 'pages', 'rating', 'reviews', 'title', 'totalratings']


In [35]:
print("\nMissing values:")
print(data.isna().sum())

print("\nData types:")
print(data.dtypes)

print("\nFirst rows:")
print(data.head())


Missing values:
author              0
bookformat       3228
desc             6772
genre           10467
img              3045
isbn            14482
isbn13          11435
link                0
pages               0
rating              0
reviews             0
title               1
totalratings        0
dtype: int64

Data types:
author              str
bookformat          str
desc                str
genre               str
img                 str
isbn                str
isbn13              str
link                str
pages             int64
rating          float64
reviews           int64
title               str
totalratings      int64
dtype: object

First rows:
                             author bookformat  \
0              Laurence M. Hauptman  Hardcover   
1  Charlotte Fiell,Emmanuelle Dirix  Paperback   
2                     Andy Anderson  Paperback   
3              Carlotta R. Anderson  Hardcover   
4                     Jean Leveille        NaN   

                               

In [36]:
editions = data.copy()

In [37]:
editions["bookformat_clean"] = (
    editions["bookformat"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

In [38]:
excluded_formats = (
    "kindle",
    "ebook",
    "e-book",
    "electronic",
    "audiobook",
    "audio book",
    "audible",
    "audio cd",
    "mp3 cd"
)

format_pattern = "|".join(
    re.escape(x)
    for x in excluded_formats
)

editions["is_excluded_format"] = (
    editions["bookformat_clean"]
    .str.contains(
        format_pattern,
        case=False,
        na=False
    )
)

In [39]:
print(
    "Books/editions before format filtering:",
    len(editions)
)

print(
    "Excluded Kindle/eBook/Audiobook records:",
    editions["is_excluded_format"].sum())

Books/editions before format filtering: 100000
Excluded Kindle/eBook/Audiobook records: 10351


In [40]:
editions = editions[
    ~editions["is_excluded_format"]
].copy()

In [41]:
def normalize_title(title):

    if pd.isna(title):
        return ""

    title = str(title).lower().strip()

    # Remove common edition indicators at the END of titles.
    title = re.sub(
        r"\s*\((?:"
        r"kindle edition|"
        r"kindle|"
        r"ebook|"
        r"e-book|"
        r"audiobook|"
        r"audio book|"
        r"audio|"
        r"paperback|"
        r"hardcover|"
        r"mass market paperback|"
        r"mass-market paperback|"
        r"illustrated edition|"
        r"anniversary edition|"
        r"edition"
        r")\)\s*$",
        "",
        title,
        flags=re.IGNORECASE
    )

    # Remove punctuation
    title = re.sub(r"[^a-z0-9\s]", " ", title)

    # Remove extra spaces
    title = re.sub(r"\s+", " ", title)

    return title.strip()


editions["title_normalized"] = (
    editions["title"]
    .apply(normalize_title)
)


In [42]:
def normalize_author(author):

    if pd.isna(author):
        return ""

    author = str(author).lower().strip()

    author = re.sub(r"[^a-z0-9\s]", " ", author)
    author = re.sub(r"\s+", " ", author)

    return author.strip()


editions["author_normalized"] = (
    editions["author"]
    .apply(normalize_author)
)

In [43]:
editions["work_key"] = (
    editions["title_normalized"]
    + "||"
    + editions["author_normalized"]
)


print(
    "\nUnique editions:",
    len(editions)
)

print(
    "Unique title/author works:",
    editions["work_key"].nunique()
)


Unique editions: 89649
Unique title/author works: 88405


In [44]:
editions["bookformat_clean"] = (
    editions["bookformat"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

In [45]:
zero_pages = editions[editions["pages"] == 0]

print("\nBooks with 0 pages:", len(zero_pages))

print("\nFormats of 0-page books:")
print(
    zero_pages["bookformat"]
    .value_counts(dropna=False)
)


Books with 0 pages: 6316

Formats of 0-page books:
bookformat
Paperback                2885
NaN                      1500
Hardcover                1275
Nook                      214
Mass Market Paperback     126
                         ... 
Hard cover                  1
Perfect Paperback           1
Serial Fiction              1
Booklet                     1
Wattpad                     1
Name: count, Length: 76, dtype: int64


In [46]:
editions["pages_clean"] = (
    pd.to_numeric(
        editions["pages"],
        errors="coerce"
    )
)

# 0 is not a meaningful page count.
# We preserve the original value in "pages".
editions.loc[
    editions["pages_clean"] == 0,
    "pages_clean"
] = np.nan

In [47]:
def length_category(pages):

    if pd.isna(pages):
        return "Unknown"

    if pages < 150:
        return "Very Short"

    elif pages < 250:
        return "Short"

    elif pages < 400:
        return "Medium"

    elif pages < 600:
        return "Long"

    else:
        return "Very Long"


editions["length_category"] = (
    editions["pages_clean"]
    .apply(length_category)
)

In [48]:
editions["rating_clean"] = (
    pd.to_numeric(
        editions["rating"],
        errors="coerce"
    )
)

# A Goodreads rating of 0 is treated as missing.
editions.loc[
    editions["rating_clean"] == 0,
    "rating_clean"
] = np.nan


In [49]:
editions["totalratings"] = pd.to_numeric(
    editions["totalratings"],
    errors="coerce"
)

editions["totalratings"] = (
    editions["totalratings"]
    .fillna(0)
)


# Log transformation prevents very popular books
# from dominating the numerical feature space.

editions["log_totalratings"] = np.log1p(
    editions["totalratings"]
)

In [50]:
def clean_genres(value):

    if pd.isna(value):
        return []

    genres = [
        g.strip()
        for g in str(value).split(",")
    ]

    # Remove empty values
    genres = [
        g for g in genres
        if g
    ]

    # Remove duplicates while preserving order
    genres = list(
        dict.fromkeys(genres)
    )

    return genres


editions["genre_list"] = (
    editions["genre"]
    .apply(clean_genres)
)

In [51]:
def combine_genres(series):

    combined = []

    for genre_list in series:

        for genre in genre_list:

            if genre not in combined:
                combined.append(genre)

    return combined


def first_non_null(series):

    values = series.dropna()

    if len(values) == 0:
        return np.nan

    return values.iloc[0]

In [52]:
books = (
    editions
    .groupby("work_key", as_index=False)
    .agg(
        title=("title", first_non_null),

        author=("author", first_non_null),

        title_normalized=(
            "title_normalized",
            first_non_null
        ),

        author_normalized=(
            "author_normalized",
            first_non_null
        ),

        genre_list=(
            "genre_list",
            combine_genres
        ),

        description=(
            "desc",
            first_non_null
        ),

        pages_clean=(
            "pages_clean",
            "max"
        ),

        rating_clean=(
            "rating_clean",
            "mean"
        ),

        totalratings=(
            "totalratings",
            "max"
        )
    )
)

In [53]:
books["log_totalratings"] = np.log1p(
    books["totalratings"]
)

In [54]:
books["length_category"] = (
    books["pages_clean"]
    .apply(length_category)
)


print("\nWork-level dataset:")
print(books.shape)

print(
    "\nNumber of unique works:",
    len(books)
)


Work-level dataset:
(88405, 12)

Number of unique works: 88405


In [64]:
books = books.reset_index(drop=True)

books["book_id"] = (
    books.index.astype(int)
)

print(
    "Number of books:",
    len(books)
)

print(
    books[
        [
            "book_id",
            "title",
            "author"
        ]
    ].head()
)

Number of books: 88405
   book_id                                              title  \
0        0  è­¦è¦–åºç‰¹çŠ¯èª²007 [Keishichou Tokuhanka 00...   
1        1  ãƒ‡ãƒ¼ãƒˆãƒ»ã‚¢ãƒ»ãƒ©ã‚¤ãƒ– 02 - å››ç³¸ä¹ƒãƒ‘ã...   
2        2  ãƒ‡ãƒ¼ãƒˆãƒ»ã‚¢ãƒ»ãƒ©ã‚¤ãƒ– 03 - ç‹‚ä¸‰ã‚­ãƒ©ã...   
3        3  ã‚¢ã‚¯ã‚»ãƒ«ãƒ»ãƒ¯ãƒ¼ãƒ«ãƒ‰ 04 è’¼ç©ºã¸ã®é£›...   
4        4                              04 Colours And Shapes   

                                          author  
0                           Eiri Kaji,èˆµ è‹±é‡Œ  
1  Koushi Tachibana,æ©˜ å…¬å¸,Tsunako,ã¤ãªã“  
2  Koushi Tachibana,æ©˜ å…¬å¸,Tsunako,ã¤ãªã“  
3               Reki Kawahara,Reki Kawahara,HiMA  
4                   Lynne Bradbury,Lynn N Grundy  


In [55]:
example_title = "Fahrenheit 451"

example = editions[
    editions["title"]
    .str.lower()
    .str.contains(
        example_title.lower(),
        na=False
    )
]

print("\nExample editions:")
print(
    example[
        [
            "title",
            "author",
            "bookformat",
            "pages",
            "rating",
            "totalratings"
        ]
    ].head(20)
)


Example editions:
                title                                   author bookformat  \
50246  Fahrenheit 451  Ray Bradbury,Michael Moorcock,Sam Weber  Hardcover   

       pages  rating  totalratings  
50246    176     4.1           246  


In [56]:
mlb = MultiLabelBinarizer()

genre_matrix = mlb.fit_transform(
    books["genre_list"]
).astype(np.float32)

genre_matrix = csr_matrix(
    genre_matrix
)


print(
    "\nNumber of genre/tag features:",
    genre_matrix.shape[1]
)



Number of genre/tag features: 1162


In [57]:
numeric_features = books[
    [
        "log_totalratings",
        "rating_clean",
        "pages_clean"
    ]
].copy()


# Replace missing values with median
numeric_features = (
    numeric_features
    .fillna(
        numeric_features.median()
    )
)

scaler = StandardScaler()

numeric_matrix = scaler.fit_transform(
    numeric_features
).astype(np.float32)

GENRE_WEIGHT = 0.7
POPULARITY_WEIGHT = 0.3
RATING_WEIGHT = 0.2
PAGES_WEIGHT = 0.1
genre_matrix = (
    genre_matrix * GENRE_WEIGHT
)

numeric_matrix[:, 0] *= POPULARITY_WEIGHT
numeric_matrix[:, 1] *= RATING_WEIGHT
numeric_matrix[:, 2] *= PAGES_WEIGHT


numeric_matrix = csr_matrix(
    numeric_matrix
)

In [58]:
content_matrix = hstack(
    [
        genre_matrix,
        numeric_matrix
    ]
).tocsr()


print(
    "\nFinal content matrix:",
    content_matrix.shape
)


Final content matrix: (88405, 1165)


In [59]:
model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=21,
    n_jobs=-1
)

model.fit(content_matrix)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",21
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",-1


In [60]:
def recommend_similar_books(title, n=10):

    # Find matching title
    matches = books[
        books["title"]
        .fillna("")
        .str.lower()
        == title.lower()
    ]

    if matches.empty:

        print(
            f"Book not found: {title}"
        )

        return None
    # If multiple works have exactly the same title,
    # display them rather than silently choosing one.
    if len(matches) > 1:

        print(
            "Multiple books with this title were found:"
        )

        print(
            matches[
                [
                    "title",
                    "author"
                ]
            ]
        )

        print(
            "\nPlease search using the author as well."
        )

        return None


    book_index = matches.index[0]


    # Get vector of selected book
    book_vector = content_matrix[
        book_index
    ]


    # Find nearest books
    distances, indices = model.kneighbors(
        book_vector,
        n_neighbors=n + 1
    )


    distances = distances[0]
    indices = indices[0]


    recommendations = books.iloc[
        indices
    ].copy()


    # Convert cosine distance to similarity
    recommendations["similarity"] = (
        1 - distances
    )


    # Remove the original book
    recommendations = recommendations[
        recommendations.index != book_index
    ]


    # Return requested columns
    recommendations = recommendations[
        [
            "title",
            "author",
            "genre_list",
            "pages_clean",
            "length_category",
            "rating_clean",
            "totalratings",
            "similarity"
        ]
    ]


    return recommendations.head(n)

In [61]:
data.columns

Index(['author', 'bookformat', 'desc', 'genre', 'img', 'isbn', 'isbn13',
       'link', 'pages', 'rating', 'reviews', 'title', 'totalratings'],
      dtype='str')

In [62]:
recommendations = recommend_similar_books(
    "Animal Farm",
    10
)

display(
    recommendations
)

,title,author,genre_list,pages_clean,length_category,rating_clean,totalratings,similarity
13179,Cat's Cradle,Kurt Vonnegut Jr.,"[Fiction, Classics, Science Fiction, Humor, Li...",179.0,Short,4.16,350196,0.731441
78376,The Veldt,"Ray Bradbury,Gary Kelley","[Short Stories, Science Fiction, Classics, Fic...",45.0,Very Short,4.20,7202,0.723923
66454,The Dispossessed,Ursula K. Le Guin,"[Science Fiction, Fiction, Fantasy, Dystopia, ...",387.0,Medium,4.20,96025,0.716525
77416,The Tempest,"William Shakespeare,Paul Werstine,David Lindle...","[Classics, Plays, Drama, Fiction, Fantasy, Aca...",218.0,Short,3.80,180980,0.695631
73316,The Odyssey,"Homer,Robert Fagles,Bernard Knox","[Classics, Fiction, Poetry, Fantasy, Mythology...",541.0,Long,3.78,898828,0.686296
75719,The Scarlet Letter,"Nathaniel Hawthorne,Thomas E. Connolly,Nina Baym","[Classics, Fiction, Historical, Historical Fic...",279.0,Medium,3.42,733658,0.681473
77612,The Three Stigmata of Palmer Eldritch,Philip K. Dick,"[Science Fiction, Fiction, Science Fiction Fan...",231.0,Short,4.01,33346,0.673184
23580,Fahrenheit 451,"Ray Bradbury,Michael Moorcock,Sam Weber","[Science Fiction, Classics, Fiction, Dystopia,...",176.0,Short,4.10,246,0.664565
47708,Ordinary People,Judith Guest,"[Fiction, Classics, Contemporary, Psychology, ...",263.0,Medium,3.91,19185,0.664234
34417,It Can't Happen Here,"Sinclair Lewis,Michael R. Meyer","[Fiction, Classics, Politics, Science Fiction,...",400.0,Long,3.77,15929,0.660223
